In [1]:
from aws_cdk import (
    Stack,
    aws_lambda as _lambda,
    aws_apigateway as apigw,
    aws_iam as iam
)
from constructs import Construct

In [ ]:
class CdkLambdaCiStack(Stack):

    def __init__(self, scope: Construct, construct_id: str, **kwargs):

        super().__init__(scope, construct_id, **kwargs)

        lambda_a_role = iam.Role(
            self, 
            "LambdaARole",
            assumed_by=iam.ServicePrincipal("lambda.amazonaws.com")
        )
        lambda_a_role.add_managed_policy(
            iam.ManagedPolicy.from_aws_managed_policy_name("service-role/AWSLambdaBasicExecutionRole")
        )
        lambda_a_role.add_to_policy(
            iam.PolicyStatement(
                actions=["sagemaker:InvokeEndpoint"],
                resources=["arn:aws:sagemaker:us-east-1:123456789012:endpoint/*"]
            )
        )


        lambda_b_role = iam.Role(
            self,
            "LambdaBRole",
            assumed_by=iam.ServicePrincipal("lambda.amazonaws.com")
        )
        lambda_b_role.add_managed_policy(
            iam.ManagedPolicy.from_aws_managed_policy_name("service-role/AWSLambdaBasicExecutionRole")
        )
        lambda_b_role.add_to_policy(
            iam.PolicyStatement(
                actions=[
                    "sagemaker:CreateEndpoint",
                    "sagemaker:CreateEndpointConfig",
                    "sagemaker:DescribeModel"
                ],
                resources=["*"]
            )
        )

        # Lambda A: expuesta por API Gateway
        lambda_a = _lambda.Function(
            self,
            "InvokeInference",
            runtime=_lambda.Runtime.PYTHON_3_11,
            handler="app.lambda_handler",
            code=_lambda.Code.from_asset("lambda_functions/invoke_inference"),
            role=lambda_a_role
        )

        # Lambda B: usada en pipeline
        lambda_b = _lambda.Function(
            self,
            "CreateEndpoint",
            runtime=_lambda.Runtime.PYTHON_3_11,
            handler="app.lambda_handler",
            code=_lambda.Code.from_asset("lambda_functions/create_endpoint"),
            role=lambda_b_role
        )

        # API Gateway solo para Lambda A
        api = apigw.RestApi(self, "MLInferenceAPI")
        api.root.add_resource("predict").add_method(
            "POST", apigw.LambdaIntegration(lambda_a)
        )